# 01 — Inventaire des sources (jury data scientist)

**Objectif :** charger les 11 CSV du sujet en DataFrames, vérifier les colonnes attendues, puis explorer le jeu.

> Données **synthétiques** — exercice pédagogique, jamais une décision clinique réelle.

**Parcours :** exécutez les cellules dans l’ordre (**Run All**).


## 1. Imports & chemin projet

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

# Remonte jusqu’à la racine du projet (dossier contenant patients.csv + Sujet.md)
ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "patients.csv").is_file() and (p / "Sujet.md").is_file()
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

print("Racine projet :", ROOT)
print("CSV à la racine :", sorted(ROOT.glob("*.csv")))


Racine projet : /Users/segalini-briant/Documents/GitHub/CISIA - Sujet Santé
CSV à la racine : [PosixPath('/Users/segalini-briant/Documents/GitHub/CISIA - Sujet Santé/actes.csv'), PosixPath('/Users/segalini-briant/Documents/GitHub/CISIA - Sujet Santé/biologies.csv'), PosixPath('/Users/segalini-briant/Documents/GitHub/CISIA - Sujet Santé/comptes_rendus.csv'), PosixPath('/Users/segalini-briant/Documents/GitHub/CISIA - Sujet Santé/diagnostics.csv'), PosixPath('/Users/segalini-briant/Documents/GitHub/CISIA - Sujet Santé/historique.csv'), PosixPath('/Users/segalini-briant/Documents/GitHub/CISIA - Sujet Santé/medications.csv'), PosixPath('/Users/segalini-briant/Documents/GitHub/CISIA - Sujet Santé/objets_connectes.csv'), PosixPath('/Users/segalini-briant/Documents/GitHub/CISIA - Sujet Santé/patients.csv'), PosixPath('/Users/segalini-briant/Documents/GitHub/CISIA - Sujet Santé/sejours.csv'), PosixPath('/Users/segalini-briant/Documents/GitHub/CISIA - Sujet Santé/signes_vitaux.csv'), 

## 2. Schéma attendu (colonnes obligatoires)

Chaque fichier doit contenir **exactement** ces colonnes (ordre libre, noms stricts).

In [2]:
SCHEMA_ATTENDU: dict[str, list[str]] = {
    "patients.csv": [
        "PatientID", "NomPrenom", "DateNaissance", "Sexe", "CodePostal", "Commune",
        "RegimeAssurance", "MedecinTraitant", "PersonneAPrevenir", "SituationFamiliale",
        "PathologiesChroniques",
    ],
    "sejours.csv": [
        "SejourID", "PatientID", "DateAdmission", "DateSortie", "DureeSejour", "Service",
        "TypeSejour", "GHM", "ModeSortie", "Readmission30j",
    ],
    "historique.csv": [
        "EvenementID", "PatientID", "TypeEvenement", "DateEvenement", "DureeJours", "CodeEtablissement",
    ],
    "diagnostics.csv": [
        "DiagnosticID", "SejourID", "CodeCIM10", "LibelleDiagnostic", "Role",
    ],
    "actes.csv": [
        "ActeID", "SejourID", "CodeCCAM", "LibelleActe", "DateActe", "CodeExecutant",
    ],
    "biologies.csv": [
        "BiologieID", "SejourID", "DatePrelevement", "Panel", "Valeur", "Unite",
        "ValeurReferenceBas", "ValeurReferenceHaut",
    ],
    "signes_vitaux.csv": [
        "ConstanteID", "SejourID", "Horodatage", "FrequenceCardiaque", "TensionSystolique",
        "TensionDiastolique", "Temperature", "FrequenceRespiratoire", "SpO2",
    ],
    "medications.csv": [
        "PrescriptionID", "PatientID", "SejourID", "CodeATC", "LibelleMedicament",
        "Posologie", "DateDebut", "DateFin", "Voie",
    ],
    "comptes_rendus.csv": [
        "CompteRenduID", "SejourID", "TypeCR", "DateCR", "TexteCR",
    ],
    "objets_connectes.csv": [
        "MesureID", "PatientID", "Horodatage", "TypeMesure", "Valeur", "QualiteSignal",
    ],
    "territoire_insee.csv": [
        "Commune", "CodePostal", "IndiceDefavorisation", "DensiteMedicale", "PopulationCommune",
    ],
}

print(f"{len(SCHEMA_ATTENDU)} fichiers attendus")


11 fichiers attendus


## 3. Chargement CSV → DataFrames

In [3]:
dfs: dict[str, pd.DataFrame] = {}
erreurs_chargement: list[str] = []

for fichier in SCHEMA_ATTENDU:
    chemin = ROOT / fichier
    if not chemin.is_file():
        erreurs_chargement.append(f"MANQUANT : {fichier}")
        continue
    dfs[fichier] = pd.read_csv(chemin)
    print(f"✓ {fichier:25s} → {len(dfs[fichier]):>7,} lignes × {dfs[fichier].shape[1]} colonnes")

if erreurs_chargement:
    raise FileNotFoundError("\n".join(erreurs_chargement))

print(f"\n{len(dfs)} DataFrames chargés.")


✓ patients.csv              →     620 lignes × 11 colonnes
✓ sejours.csv               →     878 lignes × 10 colonnes
✓ historique.csv            →     759 lignes × 6 colonnes
✓ diagnostics.csv           →   2,062 lignes × 5 colonnes
✓ actes.csv                 →   1,254 lignes × 6 colonnes
✓ biologies.csv             →   3,721 lignes × 8 colonnes
✓ signes_vitaux.csv         →  24,916 lignes × 9 colonnes
✓ medications.csv           →   3,728 lignes × 9 colonnes
✓ comptes_rendus.csv        →   1,144 lignes × 5 colonnes
✓ objets_connectes.csv      →   6,297 lignes × 6 colonnes
✓ territoire_insee.csv      →      65 lignes × 5 colonnes

11 DataFrames chargés.


## 4. Vérification des colonnes

On contrôle : colonnes **manquantes**, colonnes **en trop**, et on affiche un tableau de conformité.

In [4]:
rapport = []
for fichier, attendues in SCHEMA_ATTENDU.items():
    df = dfs[fichier]
    cols = set(df.columns)
    attendues_set = set(attendues)
    manquantes = sorted(attendues_set - cols)
    en_trop = sorted(cols - attendues_set)
    ok = not manquantes and not en_trop
    rapport.append({
        "fichier": fichier,
        "lignes": len(df),
        "nb_colonnes": len(df.columns),
        "conforme": ok,
        "manquantes": ", ".join(manquantes) or "—",
        "en_trop": ", ".join(en_trop) or "—",
    })

rapport_df = pd.DataFrame(rapport)
display(rapport_df)

nb_ko = (~rapport_df["conforme"]).sum()
if nb_ko:
    raise AssertionError(f"{nb_ko} fichier(s) non conforme(s) — corrigez avant de continuer.")
print("✓ Toutes les colonnes attendues sont présentes.")


,fichier,lignes,nb_colonnes,conforme,manquantes,en_trop
0,patients.csv,620,11,True,—,—
1,sejours.csv,878,10,True,—,—
2,historique.csv,759,6,True,—,—
3,diagnostics.csv,2062,5,True,—,—
4,actes.csv,1254,6,True,—,—
5,biologies.csv,3721,8,True,—,—
6,signes_vitaux.csv,24916,9,True,—,—
7,medications.csv,3728,9,True,—,—
8,comptes_rendus.csv,1144,5,True,—,—
9,objets_connectes.csv,6297,6,True,—,—


✓ Toutes les colonnes attendues sont présentes.


## 5. Vue d’ensemble (inventaire)

In [5]:
inventaire = pd.DataFrame([
    {
        "fichier": nom,
        "lignes": len(df),
        "colonnes": df.shape[1],
        "memoire_Mo": round(df.memory_usage(deep=True).sum() / 1e6, 2),
        "doublons_lignes": int(df.duplicated().sum()),
        "aperçu_colonnes": ", ".join(df.columns[:6].tolist()),
    }
    for nom, df in dfs.items()
]).sort_values("lignes", ascending=False)

inventaire


,fichier,lignes,colonnes,memoire_Mo,doublons_lignes,aperçu_colonnes
6,signes_vitaux.csv,24916,9,5.81,0,"ConstanteID, SejourID, Horodatage, FrequenceCa..."
9,objets_connectes.csv,6297,6,1.92,0,"MesureID, PatientID, Horodatage, TypeMesure, V..."
7,medications.csv,3728,9,1.95,0,"PrescriptionID, PatientID, SejourID, CodeATC, ..."
5,biologies.csv,3721,8,1.20,37,"BiologieID, SejourID, DatePrelevement, Panel, ..."
3,diagnostics.csv,2062,5,0.63,20,"DiagnosticID, SejourID, CodeCIM10, LibelleDiag..."
4,actes.csv,1254,6,0.47,0,"ActeID, SejourID, CodeCCAM, LibelleActe, DateA..."
8,comptes_rendus.csv,1144,5,0.54,0,"CompteRenduID, SejourID, TypeCR, DateCR, TexteCR"
1,sejours.csv,878,10,0.44,0,"SejourID, PatientID, DateAdmission, DateSortie..."
2,historique.csv,759,6,0.23,0,"EvenementID, PatientID, TypeEvenement, DateEve..."
0,patients.csv,620,11,0.37,0,"PatientID, NomPrenom, DateNaissance, Sexe, Cod..."


## 6. Aperçu des tables principales

Variables exposées pour la suite du notebook (et des notebooks suivants) :
`patients`, `sejours`, `diagnostics`, `actes`, `biologies`, `signes_vitaux`, `medications`,
`comptes_rendus`, `objets_connectes`, `historique`, `territoire`.

In [6]:
patients = dfs["patients.csv"]
sejours = dfs["sejours.csv"]
historique = dfs["historique.csv"]
diagnostics = dfs["diagnostics.csv"]
actes = dfs["actes.csv"]
biologies = dfs["biologies.csv"]
signes_vitaux = dfs["signes_vitaux.csv"]
medications = dfs["medications.csv"]
comptes_rendus = dfs["comptes_rendus.csv"]
objets_connectes = dfs["objets_connectes.csv"]
territoire = dfs["territoire_insee.csv"]

print("patients"); display(patients.head(3))
print("sejours"); display(sejours.head(3))


patients


,PatientID,NomPrenom,DateNaissance,Sexe,CodePostal,Commune,RegimeAssurance,MedecinTraitant,PersonneAPrevenir,SituationFamiliale,PathologiesChroniques
0,PAT-00001,Martin Monique,1927-02-12,F,97412,Bras-Panon,MSA,MED-056,Fils Martin,Vit seul,InsuffisanceRenale|InsuffisanceCardiaque|Hyper...
1,PAT-00002,Petit Christine,1974-05-24,F,97424,Cilaos,CPAM,MED-027,Fils Brun,Vit seul,Aucune
2,PAT-00003,Guerin Chantal,1947-11-12,F,23000,Guéret,CPAM,MED-055,Fils Leroy,Vit seul,Aucune


sejours


,SejourID,PatientID,DateAdmission,DateSortie,DureeSejour,Service,TypeSejour,GHM,ModeSortie,Readmission30j
0,SEJ-000001,PAT-00001,2023-12-20 02:27:06,2023-12-26 02:27:06,6,Pneumologie,Urgent,04M05,Domicile,1
1,SEJ-000002,PAT-00002,2023-10-06 17:15:58,2023-10-09 17:15:58,3,Chirurgie,Urgent,09C08,Domicile,0
2,SEJ-000003,PAT-00003,2023-12-31 04:22:36,2024-01-03 04:22:36,3,Urgences,Programmé,01M02,Deces,0


## 7. Premières analyses métier (séjours)

In [7]:
# Modes de sortie & cible de réadmission
print("ModeSortie :")
display(sejours["ModeSortie"].value_counts(dropna=False))

domicile = sejours.loc[sejours["ModeSortie"] == "Domicile"].copy()
print(f"\nSéjours sortie Domicile : {len(domicile):,} / {len(sejours):,}")

if "Readmission30j" in domicile.columns:
    prev = domicile["Readmission30j"].mean()
    print(f"Prévalence Readmission30j (Domicile) : {prev:.1%}")
    display(domicile["Readmission30j"].value_counts(dropna=False))


ModeSortie :


ModeSortie
Domicile     480
Transfert    144
Deces         92
EHPAD         87
HAD           75
Name: count, dtype: int64


Séjours sortie Domicile : 480 / 878
Prévalence Readmission30j (Domicile) : 18.5%


Readmission30j
0    391
1     89
Name: count, dtype: int64

In [8]:
# Jointure patient ↔ séjour (aperçu)
base = sejours.merge(
    patients[["PatientID", "Sexe", "DateNaissance", "Commune", "CodePostal"]],
    on="PatientID",
    how="left",
    validate="many_to_one",
)
print(f"Lignes après jointure : {len(base):,} (séjours={len(sejours):,})")
print("Patients sans match :", base["Sexe"].isna().sum())
display(base.head(5))


Lignes après jointure : 878 (séjours=878)
Patients sans match : 0


,SejourID,PatientID,DateAdmission,DateSortie,DureeSejour,Service,TypeSejour,GHM,ModeSortie,Readmission30j,Sexe,DateNaissance,Commune,CodePostal
0,SEJ-000001,PAT-00001,2023-12-20 02:27:06,2023-12-26 02:27:06,6,Pneumologie,Urgent,04M05,Domicile,1,F,1927-02-12,Bras-Panon,97412
1,SEJ-000002,PAT-00002,2023-10-06 17:15:58,2023-10-09 17:15:58,3,Chirurgie,Urgent,09C08,Domicile,0,F,1974-05-24,Cilaos,97424
2,SEJ-000003,PAT-00003,2023-12-31 04:22:36,2024-01-03 04:22:36,3,Urgences,Programmé,01M02,Deces,0,F,1947-11-12,Guéret,23000
3,SEJ-000004,PAT-00004,2023-07-28 23:23:19,2023-08-18 23:23:19,21,Urgences,Urgent,24Z08,Domicile,0,F,1948-12-31,Rodez,12000
4,SEJ-000005,PAT-00005,2023-09-18 11:50:09,2023-09-20 11:50:09,2,SoinsContinus,Urgent,25Z03,Transfert,0,F,1946-06-29,Entre-Deux,97414


## 8. Cardinalités & clés

In [9]:
cles = {
    "patients.csv": "PatientID",
    "sejours.csv": "SejourID",
    "diagnostics.csv": "DiagnosticID",
    "actes.csv": "ActeID",
    "biologies.csv": "BiologieID",
    "signes_vitaux.csv": "ConstanteID",
    "medications.csv": "PrescriptionID",
    "comptes_rendus.csv": "CompteRenduID",
    "objets_connectes.csv": "MesureID",
    "historique.csv": "EvenementID",
}

rows = []
for fichier, cle in cles.items():
    df = dfs[fichier]
    rows.append({
        "fichier": fichier,
        "clé": cle,
        "n_unique": df[cle].nunique(dropna=False),
        "n_lignes": len(df),
        "unicité_ok": df[cle].nunique(dropna=False) == len(df),
        "nulls": int(df[cle].isna().sum()),
    })

pd.DataFrame(rows)


,fichier,clé,n_unique,n_lignes,unicité_ok,nulls
0,patients.csv,PatientID,620,620,True,0
1,sejours.csv,SejourID,878,878,True,0
2,diagnostics.csv,DiagnosticID,2042,2062,False,0
3,actes.csv,ActeID,1254,1254,True,0
4,biologies.csv,BiologieID,3684,3721,False,0
5,signes_vitaux.csv,ConstanteID,24916,24916,True,0
6,medications.csv,PrescriptionID,3728,3728,True,0
7,comptes_rendus.csv,CompteRenduID,1144,1144,True,0
8,objets_connectes.csv,MesureID,6297,6297,True,0
9,historique.csv,EvenementID,759,759,True,0


## 9. Suite du parcours

| Étape | Notebook |
|-------|----------|
| Qualité / hallucinations | [`02_qualite_hallucinations.ipynb`](02_qualite_hallucinations.ipynb) |
| Registre RGPD | [`03_registre_rgpd.ipynb`](03_registre_rgpd.ipynb) |
| Pipeline features | [`04_pipeline_donnees.ipynb`](04_pipeline_donnees.ipynb) |
| Benchmark modèles | [`../02_modeles/01_benchmark_optuna.ipynb`](../02_modeles/01_benchmark_optuna.ipynb) |

Les DataFrames `patients`, `sejours`, … restent disponibles dans ce kernel tant qu’il reste ouvert.
